In [ ]:
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.distributions as D
from torch.utils.data import TensorDataset, DataLoader

# --- 1. Experiment Environment Setup ---
sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device used: {device}")

# Hyperparameters
N_SAMPLES = 5000
BATCH_SIZE = 256
LEARNING_RATE = 1e-2
N_ITERATIONS_SFT_KD = 2000
N_ITERATIONS_ALIGN = 2000 # Default fixed alignment iterations
N_TRIALS_PER_SETTING = 20
KD_SAMPLING_BETA = 1.25

# Number of samples for Knowledge Distillation (KD)
N_SAMPLES_KD = 2000

PPO_CLIP_EPS = 0.2

# --- Experiment Variable Configuration ---
ALIGN_ITERATIONS_arr = [1500, 2000, 2500, 3000]
BETA_GRPO_FIXED = 1.0 # Fixed Beta value for GRPO

# Mode configuration
N_GT_MODES = 8
N_SFT_MODES = 6
N_FINAL_MODES_arr = [4, 3] # Comparing different final mode counts

# --- Target Mode Configuration ---
# Changed from single target (7) to multiple targets (6, 7)
TARGET_INDICES = [6, 7]

# --- 2. Model and Utility Functions ---
true_centers = np.array([[-1.5, 1.5], [0, 1.5], [1.5, 1.5], [-1.5, 0], [1.5, 0], [-1.5, -1.5], [0, -1.5], [1.5, -1.5]], dtype=np.float32)
true_centers_tensor = torch.from_numpy(true_centers).to(device)
true_cov = torch.eye(2, device=device) * 0.05
mix = D.Categorical(torch.ones(8, device=device)); comp = D.Independent(D.Normal(true_centers_tensor, true_cov.diag().sqrt()), 1)
ground_truth_dist = D.MixtureSameFamily(mix, comp)
gt_validation_set = ground_truth_dist.sample((N_SAMPLES,)).to(device)

# --- Multi-Target Distribution Setup ---
TARGET_CENTERS = true_centers_tensor[TARGET_INDICES] # Shape: (2, 2)

# Validation target distribution: Mixture of modes 6 and 7
target_mix = D.Categorical(torch.ones(len(TARGET_INDICES), device=device))
target_comp = D.Independent(D.Normal(TARGET_CENTERS, true_cov.diag().sqrt()), 1)
main_target_mode_dist = D.MixtureSameFamily(target_mix, target_comp)

class MoGModel(nn.Module):
    def __init__(self, n_components):
        super().__init__(); self.n_components = n_components
        self.mix_logits = nn.Parameter(torch.randn(n_components)); self.mus = nn.Parameter(torch.randn(n_components, 2)); self.log_diags = nn.Parameter(torch.randn(n_components, 2))
    def get_distribution(self):
        mix = D.Categorical(logits=self.mix_logits); comp = D.Independent(D.Normal(self.mus, self.log_diags.exp()), 1)
        return D.MixtureSameFamily(mix, comp)
    def log_prob(self, x): return self.get_distribution().log_prob(x)
        
# ValueModel is unused in GRPO but retained for structural compatibility
class ValueModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.value = nn.Parameter(torch.randn(1))
    def forward(self, num_samples):
        return self.value.expand(num_samples)

@torch.no_grad()
def calculate_precision_recall(model_dist, gt_dist, gt_validation_set):
    model_samples = model_dist.sample((gt_validation_set.shape[0],)); precision = gt_dist.log_prob(model_samples).mean().item()
    recall = model_dist.log_prob(gt_validation_set).mean().item()
    return precision, recall

@torch.no_grad()
def calculate_target_precision(model_dist, target_dist, n_samples=2000):
    model_samples = model_dist.sample((n_samples,)); return target_dist.log_prob(model_samples).mean().item()

# --- Reward Function for Multiple Targets ---
def get_perfect_reward_signal(batch, target_centers):
    """
    batch: (Batch_Size, 2)
    target_centers: (N_Targets, 2) -> here (2, 2)
    """
    # Broadcasting: (Batch, 1, 2) - (N_Targets, 2) -> (Batch, N_Targets, 2)
    diff = batch.unsqueeze(1) - target_centers 
    
    # Calculate squared distances to each target -> (Batch, N_Targets)
    distances_sq = torch.sum(diff**2, dim=2) 
    
    # Calculate Gaussian rewards -> (Batch, N_Targets)
    rewards_all = 10.0 * torch.exp(-distances_sq * 2.0)
    
    # Select max reward (closest target) for each sample -> (Batch,)
    return rewards_all.max(dim=1).values

# --- 3. Pipeline Comparison Experiment ---
results = []
start_time = time.time()
print(f"--- Running GRPO Pipeline Comparison across Alignment Iterations (PPO_CLIP_EPS fixed at {PPO_CLIP_EPS}) ---")
print(f"--- Targeting Modes: {TARGET_INDICES} ---")
print(f"--- Using KD Sampling Beta = {KD_SAMPLING_BETA} ---")

for n_modes in N_FINAL_MODES_arr:
    print(f"\n{'#'*80}\n### TESTING WITH N_FINAL_MODES = {n_modes} ###\n{'#'*80}")
    
    # Main loop: Iterating over Alignment Iterations
    for align_iter in ALIGN_ITERATIONS_arr:
        print(f"\n{'='*80}\n--- Testing ALIGN_ITERATIONS = {align_iter} ({N_TRIALS_PER_SETTING} trials) ---\n{'='*80}")
        for trial in range(N_TRIALS_PER_SETTING):
            
            # Train SFT (Supervised Fine-Tuning) model
            sft_data = ground_truth_dist.sample((N_SAMPLES,)).to(device)
            sft_dataloader = DataLoader(TensorDataset(sft_data), batch_size=BATCH_SIZE, shuffle=True)
            sft_model = MoGModel(n_components=N_SFT_MODES).to(device)
            optimizer_sft = torch.optim.Adam(sft_model.parameters(), lr=LEARNING_RATE)
            for i in range(N_ITERATIONS_SFT_KD):
                for (batch,) in sft_dataloader:
                    optimizer_sft.zero_grad(); loss = -sft_model.log_prob(batch).mean(); loss.backward(); optimizer_sft.step()
            sft_dist = sft_model.get_distribution()
        
            # --- Pipeline A: KD -> Align ---
            print(f"  Trial {trial+1}/{N_TRIALS_PER_SETTING} | Pipeline A (KD -> Align)...")
            # A-1: Knowledge Distillation
            kd_model_a = MoGModel(n_components=n_modes).to(device)
            optimizer_kd_a = torch.optim.Adam(kd_model_a.parameters(), lr=LEARNING_RATE)

            with torch.no_grad():
                modified_logits = sft_dist.mixture_distribution.logits * KD_SAMPLING_BETA
                modified_mix = D.Categorical(logits=modified_logits)
                modified_teacher_dist = D.MixtureSameFamily(modified_mix, sft_dist.component_distribution)

            for i in range(N_ITERATIONS_SFT_KD):
                with torch.no_grad(): batch = modified_teacher_dist.sample((BATCH_SIZE,))
                with torch.no_grad(): teacher_log_probs = sft_dist.log_prob(batch)
                student_log_probs = kd_model_a.log_prob(batch)
                loss = nn.functional.kl_div(student_log_probs, teacher_log_probs, log_target=True, reduction='batchmean')
                optimizer_kd_a.zero_grad(); loss.backward(); optimizer_kd_a.step()
            kd_dist_a = kd_model_a.get_distribution()
        
            # A-2: Alignment (GRPO)
            actor_a = MoGModel(n_components=n_modes).to(device); actor_a.load_state_dict(kd_model_a.state_dict())
            # Optimizer updates only Actor parameters (Critic-free)
            optimizer_a = torch.optim.Adam(actor_a.parameters(), lr=LEARNING_RATE/1)
            
            # Train for varying alignment iterations
            for i in range(align_iter):
                with torch.no_grad(): batch = actor_a.get_distribution().sample((BATCH_SIZE,))
                if batch.size(0) != BATCH_SIZE: continue
                
                # Pass TARGET_CENTERS for reward calculation
                rewards = get_perfect_reward_signal(batch, TARGET_CENTERS)
                
                log_prob_theta = actor_a.log_prob(batch)
                with torch.no_grad(): log_prob_ref = kd_dist_a.log_prob(batch)
                
                # --- Strict GRPO Implementation (Critic-Free Baseline + PPO-Clip Objective + KL Penalty) ---
                # 1. Calculate Advantage (using group mean as baseline)
                advantage = rewards - rewards.mean()

                # 2. Calculate probability ratio and PPO clip loss
                ratio = torch.exp(log_prob_theta - log_prob_ref)
                surr1 = ratio * advantage.detach()
                surr2 = torch.clamp(ratio, 1 - PPO_CLIP_EPS, 1 + PPO_CLIP_EPS) * advantage.detach()
                loss_actor = -torch.min(surr1, surr2).mean()

                # 3. Calculate KL penalty term
                kl_penalty = log_prob_theta - log_prob_ref
                
                # 4. Final Loss: Sum of PPO-Clip loss and KL penalty
                loss = loss_actor + BETA_GRPO_FIXED * kl_penalty.mean()
                
                optimizer_a.zero_grad(); loss.backward(); optimizer_a.step()
            
            final_dist_a = actor_a.get_distribution()
            p_a, r_a = calculate_precision_recall(final_dist_a, ground_truth_dist, gt_validation_set)
            tp_a = calculate_target_precision(final_dist_a, main_target_mode_dist)
            
            with torch.no_grad():
                post_rl_samples_a = final_dist_a.sample((N_SAMPLES,))
                # Pass TARGET_CENTERS for reward calculation
                post_rl_avg_reward_a = get_perfect_reward_signal(post_rl_samples_a, TARGET_CENTERS).mean().item()
            
            print(f"    -> Result A: Precision={p_a:.3f}, Recall={r_a:.3f}, Target Precision={tp_a:.3f}, Reward={post_rl_avg_reward_a:.3f}")
            
            # Store results (using align_iterations as key)
            results.append({'pipeline': 'A: KD -> Align', 'n_modes': n_modes, 'align_iterations': align_iter, 'precision': p_a, 'recall': r_a, 'target_precision': tp_a, 'post_rl_reward': post_rl_avg_reward_a})
        
            # --- Pipeline B: Align -> KD ---
            print(f"  Trial {trial+1}/{N_TRIALS_PER_SETTING} | Pipeline B (Align -> KD)...")
            # B-1: Alignment (GRPO on SFT model)
            actor_b_interim = MoGModel(n_components=N_SFT_MODES).to(device); actor_b_interim.load_state_dict(sft_model.state_dict())
            # Optimizer updates only Actor parameters (Critic-free)
            optimizer_b1 = torch.optim.Adam(actor_b_interim.parameters(), lr=LEARNING_RATE/1)

            # Train for varying alignment iterations
            for i in range(align_iter):
                with torch.no_grad(): batch = actor_b_interim.get_distribution().sample((BATCH_SIZE,))
                if batch.size(0) != BATCH_SIZE: continue
                
                # Pass TARGET_CENTERS for reward calculation
                rewards = get_perfect_reward_signal(batch, TARGET_CENTERS)
                
                log_prob_theta = actor_b_interim.log_prob(batch)
                with torch.no_grad(): log_prob_ref = sft_dist.log_prob(batch)
                
                # --- Strict GRPO Implementation (Critic-Free Baseline + PPO-Clip Objective + KL Penalty) ---
                # 1. Calculate Advantage (using group mean as baseline)
                advantage = rewards - rewards.mean()

                # 2. Calculate probability ratio and PPO clip loss
                ratio = torch.exp(log_prob_theta - log_prob_ref)
                surr1 = ratio * advantage.detach()
                surr2 = torch.clamp(ratio, 1 - PPO_CLIP_EPS, 1 + PPO_CLIP_EPS) * advantage.detach()
                loss_actor = -torch.min(surr1, surr2).mean()

                # 3. Calculate KL penalty term
                kl_penalty = log_prob_theta - log_prob_ref

                # 4. Final Loss: Sum of PPO-Clip loss and KL penalty
                loss = loss_actor + BETA_GRPO_FIXED * kl_penalty.mean()

                optimizer_b1.zero_grad(); loss.backward(); optimizer_b1.step()
            aligned_dist_b = actor_b_interim.get_distribution()
        
            # B-2: Knowledge Distillation
            final_model_b = MoGModel(n_components=n_modes).to(device)
            optimizer_b2 = torch.optim.Adam(final_model_b.parameters(), lr=LEARNING_RATE)

            with torch.no_grad():
                modified_logits = aligned_dist_b.mixture_distribution.logits * KD_SAMPLING_BETA
                modified_mix = D.Categorical(logits=modified_logits)
                modified_teacher_dist = D.MixtureSameFamily(modified_mix, aligned_dist_b.component_distribution)

            for i in range(N_ITERATIONS_SFT_KD):
                with torch.no_grad(): batch = modified_teacher_dist.sample((BATCH_SIZE,))
                with torch.no_grad(): teacher_log_probs = aligned_dist_b.log_prob(batch)
                student_log_probs = final_model_b.log_prob(batch)
                loss = nn.functional.kl_div(student_log_probs, teacher_log_probs, log_target=True, reduction='batchmean')
                optimizer_b2.zero_grad(); loss.backward(); optimizer_b2.step()
            
            final_dist_b = final_model_b.get_distribution()
            p_b, r_b = calculate_precision_recall(final_dist_b, ground_truth_dist, gt_validation_set)
            tp_b = calculate_target_precision(final_dist_b, main_target_mode_dist)

            with torch.no_grad():
                post_rl_samples_b = final_dist_b.sample((N_SAMPLES,))
                # Pass TARGET_CENTERS for reward calculation
                post_rl_avg_reward_b = get_perfect_reward_signal(post_rl_samples_b, TARGET_CENTERS).mean().item()
                
            print(f"    -> Result B: Precision={p_b:.3f}, Recall={r_b:.3f}, Target Precision={tp_b:.3f}, Reward={post_rl_avg_reward_b:.3f}")
            
            # Store results (using align_iterations as key)
            results.append({'pipeline': 'B: Align -> KD', 'n_modes': n_modes, 'align_iterations': align_iter, 'precision': p_b, 'recall': r_b, 'target_precision': tp_b, 'post_rl_reward': post_rl_avg_reward_b})
        
        print(f"Align iterations={align_iter} complete.")

end_time = time.time()
print(f"\nTotal experiment time: {end_time - start_time:.2f} seconds")

# --- 4. Results Aggregation and Visualization ---
results_df = pd.DataFrame(results)
results_df['hue'] = results_df['pipeline'] + ' | ' + results_df['n_modes'].astype(str) + ' modes'

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle(f"Pipeline Performance vs. Alignment Iterations (GRPO, Beta={BETA_GRPO_FIXED})", fontsize=16)

# Plot 1: Final Target Precision
sns.boxplot(data=results_df, x='align_iterations', y='target_precision', hue='hue', ax=axes[0,0], showfliers=False)
axes[0,0].set_title("Final Target Precision vs. Alignment Iterations")
axes[0,0].set_xlabel("Alignment Iterations"); axes[0,0].set_ylabel("Target Precision"); axes[0,0].grid(True)
axes[0,0].legend(title='Pipeline | Modes')

# Plot 2: Final Overall Precision
sns.boxplot(data=results_df, x='align_iterations', y='precision', hue='hue', ax=axes[0,1], showfliers=False)
axes[0,1].set_title("Final Overall Precision vs. Alignment Iterations")
axes[0,1].set_xlabel("Alignment Iterations"); axes[0,1].set_ylabel("Overall Precision"); axes[0,1].grid(True)
axes[0,1].legend(title='Pipeline | Modes')

# Plot 3: Final Overall Recall
sns.boxplot(data=results_df, x='align_iterations', y='recall', hue='hue', ax=axes[1,0], showfliers=False)
axes[1,0].set_title("Final Overall Recall vs. Alignment Iterations")
axes[1,0].set_xlabel("Alignment Iterations"); axes[1,0].set_ylabel("Overall Recall"); axes[1,0].grid(True)
axes[1,0].legend(title='Pipeline | Modes')

# Plot 4: Post-RL Average Reward
sns.boxplot(data=results_df, x='align_iterations', y='post_rl_reward', hue='hue', ax=axes[1,1], showfliers=False)
axes[1,1].set_title("Final Average Reward vs. Alignment Iterations")
axes[1,1].set_xlabel("Alignment Iterations"); axes[1,1].set_ylabel("Average Reward"); axes[1,1].grid(True)
axes[1,1].legend(title='Pipeline | Modes')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()